# 从固定 ideal 基线分批注入 CW

流程只有三步：保存一套 `make_ideal` 后的基线；每个频率批次都从它深复制；每批写入自己的目录且不覆盖旧结果。

## 1. 导入

In [ ]:
from copy import deepcopy
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from pint import toa as pint_toa
from pint.logging import setup as setup_pint_logging
from pint.models import get_model
from pint.residuals import Residuals
from pta_replicator.deterministic import add_cgw
from pta_replicator.simulate import SimulatedPulsar, make_ideal, simulate_pulsar

setup_pint_logging(level="WARNING")

## 2. 参数

In [ ]:
obstime = np.arange(53000, 56650, 14)
N_PSR = 25

T_seconds = (obstime[-1] - obstime[0]) * 86400.0
f = 1.0 / T_seconds

source_ra = np.deg2rad(2.0)
source_dec = np.deg2rad(0.4)
gwphi = source_ra
gwtheta = np.pi / 2.0 - source_dec

# 增加频率时，只需在这里多写一行。
CW_BATCHES = [
    {"label": "k5p5", "k0": 5, "delta": 0.5},
]
ANALYSIS_BATCH = "k5p5"

assert len({item["label"] for item in CW_BATCHES}) == len(CW_BATCHES)
assert ANALYSIS_BATCH in {item["label"] for item in CW_BATCHES}

In [ ]:
template_par = Path("data/JPSR00.par")
data_root = Path("cw_batch_data")
model_par_dir = data_root / "model_par" / "baseline_v1"
ideal_dir = data_root / "ideal" / "baseline_v1"
cw_root = data_root / "cw"
psr_names = [f"J{i:02d}" for i in range(N_PSR)]

## 3. 生成或加载 ideal 基线

In [ ]:
def fibonacci_sphere(n):
    i = np.arange(n)
    golden_ratio = (1 + np.sqrt(5)) / 2
    z = 1 - 2 * (i + 0.5) / n
    ra = np.mod(2 * np.pi * i / golden_ratio, 2 * np.pi)
    dec = np.arcsin(z)
    return ra, dec

In [ ]:
def load_ideal_psr(parfile, timfile):
    model = get_model(parfile)

    # 传入 model，确保 PINT 使用 par 文件中的 TT(BIPM2016)。
    toas = pint_toa.get_TOAs(
        timfile, model=model, ephem="DE440", planets=True, usepickle=False
    )

    return SimulatedPulsar(
        ephem="DE440",
        model=model,
        toas=toas,
        residuals=Residuals(toas, model),
        name=model.PSR.value,
        loc={"RAJ": model.RAJ.value, "DECJ": model.DECJ.value},
        added_signals={},
        added_signals_time={},
    )

In [ ]:
ra_list, dec_list = fibonacci_sphere(N_PSR)

if not model_par_dir.exists():
    model_par_dir.mkdir(parents=True)

    for name, ra, dec in zip(psr_names, ra_list, dec_list):
        model = get_model(template_par)
        model.PSR.value = name
        model.RAJ.quantity = ra * u.rad
        model.DECJ.quantity = dec * u.rad
        model.setup()
        model.validate()
        model.write_parfile(model_par_dir / f"{name}.par")

parfiles = sorted(model_par_dir.glob("*.par"))
if [path.stem for path in parfiles] != psr_names:
    raise RuntimeError(f"{model_par_dir} 中的 par 文件不完整")

print(f"模型 par 文件：{len(parfiles)} 个")

In [ ]:
if not ideal_dir.exists():
    ideal_dir.mkdir(parents=True)

    for parfile in parfiles:
        psr = simulate_pulsar(
            parfile=str(parfile),
            obstimes=obstime,
            toaerr=0.1,
            freq=1440.0,
            observatory="AXIS",
            ephem="DE440",
        )
        make_ideal(psr)
        psr.write_partim(
            ideal_dir / f"{psr.name}.par",
            ideal_dir / f"{psr.name}.tim",
            tempo2=True,
        )
    print("ideal 基线已保存")
else:
    print("ideal 基线已存在，直接使用，不覆盖")

ideal_pars = sorted(ideal_dir.glob("*.par"))
ideal_tims = sorted(ideal_dir.glob("*.tim"))
if [p.stem for p in ideal_pars] != psr_names or [p.stem for p in ideal_tims] != psr_names:
    raise RuntimeError(f"{ideal_dir} 中的 par/tim 文件不完整")

ideal_psrs = [load_ideal_psr(par, tim) for par, tim in zip(ideal_pars, ideal_tims)]
print(f"ideal 脉冲星：{len(ideal_psrs)} 颗，每颗 {len(ideal_psrs[0].toas)} 个 TOA")

## 4. 每个频率从 ideal 独立注入

In [ ]:
def inject_cw(ideal_psrs, fgw):
    injected_psrs = deepcopy(ideal_psrs)

    for psr in injected_psrs:
        add_cgw(
            psr=psr,
            gwtheta=gwtheta,
            gwphi=gwphi,
            mc=1.0e9,
            dist=100.0,
            fgw=fgw,
            phase0=0.0,
            psi=0.0,
            inc=np.pi / 3.0,
            psrTerm=False,
            evolve=False,
            phase_approx=False,
            tref=obstime[0] * 86400.0,
            signal_name="cw",
        )

        if set(psr.added_signals) != {f"{psr.name}_cw"}:
            raise RuntimeError(f"{psr.name} 出现重复或意外信号")

    return injected_psrs

In [ ]:
ideal_mjds = {psr.name: psr.toas.get_mjds().value.copy() for psr in ideal_psrs}
batch_psrs = {}
batch_frequencies = {}

for batch in CW_BATCHES:
    label = batch["label"]
    fgw = (batch["k0"] + batch["delta"]) * f
    output_dir = cw_root / f"{label}_f{fgw:.12e}Hz"
    injected_psrs = inject_cw(ideal_psrs, fgw)

    if output_dir.exists():
        n_par = len(list(output_dir.glob("*.par")))
        n_tim = len(list(output_dir.glob("*.tim")))
        if (n_par, n_tim) != (N_PSR, N_PSR):
            raise RuntimeError(f"{output_dir} 已存在但文件不完整")
        action = "已存在，不覆盖"
    else:
        output_dir.mkdir(parents=True)
        for psr in injected_psrs:
            psr.write_partim(
                output_dir / f"{psr.name}.par",
                output_dir / f"{psr.name}.tim",
                tempo2=True,
            )
        action = "已保存"

    batch_psrs[label] = injected_psrs
    batch_frequencies[label] = fgw
    print(f"{label}: fgw={fgw:.15e} Hz，{action} -> {output_dir}")

for psr in ideal_psrs:
    assert np.array_equal(psr.toas.get_mjds().value, ideal_mjds[psr.name])
    assert psr.added_signals == {}

psrs = batch_psrs[ANALYSIS_BATCH]
fgw = batch_frequencies[ANALYSIS_BATCH]

## 5. 查看所选批次

In [ ]:
ra = np.array([psr.model.RAJ.quantity.to_value(u.rad) for psr in psrs])
dec = np.array([psr.model.DECJ.quantity.to_value(u.rad) for psr in psrs])
ra_wrapped = (ra + np.pi) % (2.0 * np.pi) - np.pi
source_ra_wrapped = (source_ra + np.pi) % (2.0 * np.pi) - np.pi

fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection="mollweide")
ax.scatter(-ra_wrapped, dec, s=30, label="Pulsars")
ax.scatter(-source_ra_wrapped, source_dec, marker="*", s=150, label="CW source")
ax.grid()
ax.legend()
ax.set_title(f"Simulated PTA sky distribution — {ANALYSIS_BATCH}")
plt.show()

In [ ]:
psr = psrs[0]
toa_mjd = psr.toas.get_mjds().value
total_residual = psr.residuals.time_resids.to_value(u.ns)
cw_residual = psr.added_signals_time[f"{psr.name}_cw"].to_value(u.ns)
difference = total_residual - cw_residual

fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
axes[0].plot(toa_mjd, total_residual, ".-")
axes[0].set_ylabel("PINT residual [ns]")
axes[1].plot(toa_mjd, cw_residual, ".-")
axes[1].set_ylabel("Injected CW [ns]")
axes[2].plot(toa_mjd, difference, ".")
axes[2].set_ylabel("PINT - CW [ns]")
axes[2].set_xlabel("MJD")
for ax in axes:
    ax.grid()
plt.tight_layout()
plt.show()

relative_rms = np.std(difference) / np.std(cw_residual)
relative_max = np.max(np.abs(difference)) / np.max(np.abs(cw_residual))
residual_amplitude = 0.5 * np.ptp(cw_residual)

print("1/T [Hz]:", f)
print("CW frequency [Hz]:", fgw)
print("max difference [ns]:", np.max(np.abs(difference)))
print("RMS difference [ns]:", np.std(difference))
print("relative RMS, relative max:", relative_rms, relative_max)
print("CW residual amplitude [ns]:", residual_amplitude)

## 6. 转为 enterprise Pulsar

In [ ]:
from enterprise.pulsar import Pulsar

enterprise_psrs = [
    Pulsar(psr.toas, psr.model, timing_package="pint", planets=False)
    for psr in psrs
]

for psr in enterprise_psrs[:3]:
    print(psr.name, len(psr.toas), psr.toas.min(), psr.toas.max())